In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [ ]:
# load
loader = PyMuPDFLoader('./data/wt_data.pdf')
docs = loader.load()    # Document 형태로 만들기 위해서 사용

In [4]:
# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = splitter.split_documents(docs)

In [ ]:
# embed
embeddings = OpenAIEmbeddings()

In [6]:
# store
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name='student_support',
    persist_directory='./student_chromadb'
)

In [7]:
# retrieve
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 5}
)

In [8]:
# test retriever
retriever.invoke('상담시간 알려주세요.')

[Document(id='26c259f7-7e8d-408e-9eea-c4083c59a80d', metadata={'creationDate': "D:20260414083437+00'00'", 'creator': '(unspecified)', 'page': 1, 'keywords': '', 'source': './data/wt_data.pdf', 'format': 'PDF 1.4', 'trapped': '', 'total_pages': 6, 'modDate': "D:20260414083437+00'00'", 'moddate': '2026-04-14T08:34:37+00:00', 'creationdate': '2026-04-14T08:34:37+00:00', 'file_path': './data/wt_data.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'author': '(anonymous)', 'title': '(anonymous)', 'subject': '(unspecified)'}, page_content='방학 중 평일\n10:00 - 17:00\n매주 수요일은 16:00 종료\n토요일\n09:30 - 12:30\n학기 중에만 운영\n일요일 및 공휴일\n운영하지 않음\n온라인 FAQ만 이용 가능\n단, 시험기간 마지막 2주에는 학업상담 수요가 증가하므로 야간상담을 화요일과 목요일에 19:30까지 한시적으로\n운영한다. 야간상담은 사전 예약자만 가능하며 당일 접수는 불가하다.\n1-2. 위치와 연락처\n학생회관 2층 204호에 위치하며, 대표전화는 02-000-1234, 대표 이메일은 support@example.ac.kr 이다.\n심리상담실은 같은 층 209호에 별도로 있다.'),
 Document(id='f086c699-7655-4048-b7a6-bfeda69331c1', metadata={'creator': '(unspecified)', 'creationdate': '2026-04-14T08:3

In [9]:
@tool
def retrieve_context(query: str) -> str:
    '''학교에 관련된 정보를 검색합니다.'''
    retrieved_docs = retriever.invoke(query)
    
    serialized = "\n\n".join(
        f"[문서 {i+1}]\n"
        f"source: {doc.metadata.get('source')}\n"
        f"page: {doc.metadata.get('page')}\n"
        f"content: {doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    )
    
    return serialized, retrieved_docs

In [10]:
agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[retrieve_context],
    system_prompt=(
        '당신은 학생지원센터 안내를 담당하고 있는 에이전트입니다.'
        '반드시 `retrieve_context` 도구를 먼저 사용하여 관련 내용을 문서에서 찾은 뒤 답변하세요.'
        '문서에 없는 내용은 추측해서 답변하지 말고 "음... 난 잘 알지 못해요"라고 답하세요.'
        '가능하면 답변 끝에 참고한 페이지 번호등의 출처를 명시하세요.'
    )
)

In [11]:
response = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "상담 예약은 언제까지 취소할 수 있나요?"
            }
        ]
    }
)

print(f"{response['messages'][-1].content}")

상담 예약 취소는 **상담 시작 4시간 전까지** 가능합니다. 그때까지 취소하면 **불이익이 없습니다**.  
참고로 **상담 시작 4시간 전 이후부터 시작 전까지** 취소하는 경우는 **지연 취소**로 분류되어 **월 2회까지 경고 없음**으로 안내되어 있어요.

출처: `./data/wt_data.pdf` **2페이지** (`2. 상담 예약 및 취소 규정`, `2-2. 취소 및 노쇼 정책`)
